In [1]:
import pandas as pd
import sys
from pathlib import Path
from model.modular import Modular
from model.solution import Solution
from model.switch import Switch
sys.path.insert(0, str(Path('.').resolve()))

vendor_paths = {
    'cisco': 'database/cisco_oferta.csv',
    'nokia': 'database/nokia_oferta.csv',
    'arista': 'database/arista_oferta.csv',
}

generated_ports_path = 'generated_ports_free.csv'

ports_df = pd.read_csv(generated_ports_path)
speed_columns = [col for col in ports_df.columns if col not in ['profile', 'role']]


def load_vendor_data(csv_path):
    df = pd.read_csv(csv_path)
    modules_df = df[df['type'] == 'modular']
    linecards_df = df[df['type'] == 'linecard']
    fixed_df = df[df['type'] == 'fixed']
    cost_by_code = (
        linecards_df.dropna(subset=['cost'])
        .drop_duplicates(subset=['code'])
        .set_index('code')['cost']
        .to_dict()
    )
    module_codes = modules_df['code'].unique()
    return df, modules_df, linecards_df, cost_by_code, module_codes, fixed_df


def build_requirement_from_row(row, speed_columns):
    requirement = pd.DataFrame([{
        'code': 'requirement',
        **{col: int(row[col]) for col in speed_columns}
    }])
    zero_cols = [col for col in speed_columns if requirement.at[0, col] == 0]
    if zero_cols:
        requirement = requirement.drop(columns=zero_cols)
    return requirement


def solve_lowest_cost(requirement, df, module_codes, cost_by_code, fixed_codes):
    best_result = None
    best_cost = None
    best_module = None
    for module_code in module_codes:
        try:
            module_data = df[df['code'] == module_code]
            module_family = module_data['family'].iloc[0]
            linecards_for_family = df[(df['type'] == 'linecard') & (df['family'] == module_family)]
            combined_data = pd.concat([module_data, linecards_for_family], ignore_index=True)

            modular = Modular(combined_data)
            solution = Solution(modular, requirement)
            result = solution.solve(heuristic="H2")

            if result is None or result.empty:
                continue

            total_cost = result['code'].map(cost_by_code).fillna(0).sum()
            if best_cost is None or total_cost < best_cost:
                best_cost = total_cost
                best_result = result
                best_module = module_code
        except Exception:
            continue
    for fixed_code in fixed_codes:
        try:
            fixed_data = df[df['code'] == fixed_code]
            if fixed_data.empty:
                continue
            fixed_switch = Switch(fixed_data)
            if not fixed_switch.check_if_satisfies(requirement):
                continue
            total_cost = fixed_switch.cost
            fixed_result = fixed_switch.obtain_max_value_configuration(requirement)
            if best_cost is None or total_cost < best_cost:
                best_cost = total_cost
                best_result = fixed_result
                best_module = fixed_code
        except Exception:
            continue
    return best_result, best_cost, best_module


summary_rows = []
for vendor, csv_path in vendor_paths.items():
    df, modules_df, linecards_df, cost_by_code, module_codes, fixed_df = load_vendor_data(csv_path)

    print(f"{vendor}: {len(modules_df)} module rows")
    print(f"{vendor}: {linecards_df['code'].nunique()} unique linecards")
    print(f"{vendor}: available modules: {modules_df['code'].unique().tolist()}")
    print(f"{vendor}: available fixed: {fixed_df['code'].unique().tolist()}")

    vendor_rows = []
    for _, row in ports_df.iterrows():
        requirement = build_requirement_from_row(row, speed_columns)
        best_result, best_cost, best_module = solve_lowest_cost(
            requirement, df, module_codes, cost_by_code, fixed_df['code'].unique()
        )

        summary = {col: int(row[col]) for col in speed_columns}
        if 'profile' in ports_df.columns:
            summary['profile'] = row['profile']
        if 'role' in ports_df.columns:
            summary['role'] = row['role']

        summary['vendor'] = vendor
        summary['best_module'] = best_module
        summary['solution_linecards'] = [] if best_result is None else best_result['code'].tolist()
        summary['solution_cost'] = None if best_cost is None else float(best_cost)

        vendor_rows.append(summary)
        summary_rows.append(summary)

    vendor_df = pd.DataFrame(vendor_rows)
    vendor_output_path = f"{vendor}_results.csv"
    vendor_df.to_csv(vendor_output_path, index=False)
    print(f"Saved {len(vendor_df)} rows to {vendor_output_path}")

summary_df = pd.DataFrame(summary_rows)

cisco: 6 module rows
cisco: 27 unique linecards
cisco: available modules: ['9808', '9804', '9516', '9508', '9504', '9400']
cisco: available fixed: ['HF6100-60L4D', 'HF6100-32D', 'HF6100-64ED', '93180YC-EX', '93108TC-EX', '93180LC-EX', '93400LD-H1', '9332D-H2R', '9364D-GX2A', '9348D-GX2A', '9332D-GX2B', 'N9K-C9316D-GX', 'N9K-C93600CD-GX', 'N9K-C9364C-GX', '93180YC-FX3', '93108TC-FX3', '93108TC-FX3P', '9348GC-FX3', '9348GC-FX3PH', '9336C-FX2', '9336C-FX2-E', '93240YC-FX2', '93360YC-FX2', '93216TC-FX2', '93180YC-FX', '93108TC-FX', '9348GC-FXP', '92348GC-X', '92348GC-FX3', '92160YC-X', '9272Q', '92304QC', '9236C', '92300YC', 'N9364E-SP2R']
Saved 1000 rows to cisco_results.csv
nokia: 3 module rows
nokia: 3 unique linecards
nokia: available modules: ['7250 IXR-6e', '7250 IXR-10e', '7250  IXR-18e']
nokia: available fixed: ['7220 IXR-H2', '7220 IXR-H4-32D', '7220 IXR-H4', '7220 IXR-H5-32D', '7220 IXR-H5-64D', '7250 IXR-X1b', '7250 IXR-X3b', '7220 IXR-D5', '7220 IXR-D4', '7220 IXR-D3L', '7220 I

In [19]:
import ast
import plotly.express as px
from IPython.display import display
import ipywidgets as widgets

result_paths = [f"{vendor}_results.csv" for vendor in vendor_paths]


def _get_speed_columns(df):
    speed_cols = []
    for col in df.columns:
        if col == "solution_cost":
            continue
        try:
            float(col)
        except (TypeError, ValueError):
            continue
        speed_cols.append(col)
    return speed_cols


def _wrap_linecards(value, items_per_line=6, max_items=24, max_chars=300):
    if pd.isna(value):
        return ""
    if isinstance(value, str):
        try:
            value = ast.literal_eval(value)
        except (SyntaxError, ValueError):
            value = [value]
    if isinstance(value, list):
        truncated = value[:max_items]
        if len(value) > max_items:
            truncated = truncated + ["..."]
        chunks = [
            ", ".join(str(item) for item in truncated[i:i + items_per_line])
            for i in range(0, len(truncated), items_per_line)
        ]
        text = "<br>".join(chunks)
    else:
        text = str(value)
    if len(text) > max_chars:
        text = text[:max_chars] + "..."
    return text


def _render_plot(result_path):
    batch_df = pd.read_csv(result_path)
    speed_columns = _get_speed_columns(batch_df)
    batch_df["total_bandwidth"] = sum(
        batch_df[col].fillna(0) * float(col) for col in speed_columns
    )

    plot_df = batch_df[batch_df["solution_cost"].notna()].copy()
    if "solution_linecards" in plot_df.columns:
        plot_df.loc[:, "solution_linecards_wrapped"] = plot_df["solution_linecards"].apply(_wrap_linecards)

    hover_cols = []
    for col in ["profile", "role", "best_module", "solution_linecards_wrapped"]:
        if col in plot_df.columns:
            hover_cols.append(col)

    for col in speed_columns:
        if col in plot_df.columns:
            hover_cols.append(col)

    def _make_plot(**thresholds):
        filtered = plot_df
        for col, min_val in thresholds.items():
            filtered = filtered[filtered[col] >= min_val]

        fig = px.scatter(
            filtered,
            x="total_bandwidth",
            y="solution_cost",
            color="role",
            hover_data=hover_cols,
            title=f"Total Bandwidth vs Solution Cost ({Path(result_path).stem})",
        )
        fig.update_layout(hoverlabel=dict(font_size=10), height=500)
        fig.update_xaxes(type="log")
        fig.update_yaxes(type="log")
        fig.write_html(f"{Path(result_path).stem}.html")
        fig.show()

    sliders = {}
    for col in speed_columns:
        if col in plot_df.columns:
            max_val = int(plot_df[col].max()) if plot_df[col].notna().any() else 0
            sliders[col] = widgets.IntSlider(
                value=0,
                min=0,
                max=max_val,
                step=1,
                description=str(col),
                continuous_update=False,
                orientation="vertical",
                layout=widgets.Layout(width="50px", height="220px"),
                style={"description_width": "0px"},
            )

    ui = widgets.HBox(
        list(sliders.values()),
        layout=widgets.Layout(
            align_items="flex-end",
            flex_flow="row wrap",
        ),
    )
    out = widgets.interactive_output(_make_plot, sliders)

    container = widgets.VBox([
        ui,
        out,
    ])

    display(container)


for result_path in result_paths:
    _render_plot(result_path)


In [18]:
vendor_paths = {
    'cisco': 'database/cisco_oferta.csv',
    'nokia': 'database/nokia_oferta.csv',
    'arista': 'database/arista_oferta.csv',
}

import pandas as pd
import sys
from pathlib import Path
from model.modular import Modular
from model.solution import Solution
from model.switch import Switch
sys.path.insert(0, str(Path('.').resolve()))
speed_columns = [col for col in ports_df.columns if col not in ['profile', 'role']]


NameError: name 'ports_df' is not defined

In [ ]:
import plotly.graph_objects as go

vendor1 = "nokia"
vendor2 = "arista"

result_paths_by_vendor = {vendor: f"{vendor}_results.csv" for vendor in vendor_paths}


def _load_results(path):
    return pd.read_csv(path)


def _build_compare_df(df1, df2, speed_columns):
    union_index = df1.index.union(df2.index)
    base = pd.DataFrame(index=union_index)

    for col in speed_columns:
        base[col] = df1[col].reindex(union_index)
        if col in df2.columns:
            base[col] = base[col].fillna(df2[col].reindex(union_index))

    info_cols = ["profile", "role", "best_module", "solution_linecards"]
    for col in info_cols:
        if col in df1.columns:
            base[f"v1_{col}"] = df1[col].reindex(union_index)
        if col in df2.columns:
            base[f"v2_{col}"] = df2[col].reindex(union_index)

    base["vendor1_cost"] = df1["solution_cost"].reindex(union_index)
    base["vendor2_cost"] = df2["solution_cost"].reindex(union_index)

    base["total_bandwidth"] = sum(
        base[col].fillna(0) * float(col) for col in speed_columns
    )
    return base


def _make_hover_text(row, vendor1, vendor2, speed_columns):
    def _vendor_block(name, cost_col, prefix):
        lines = []
        cost_val = row.get(cost_col)
        if pd.notna(cost_val):
            lines.append(f"{name} cost: {cost_val}")
        for key, label in [
            ("profile", "profile"),
            ("role", "role"),
            ("best_module", "best_module"),
        ]:
            col = f"{prefix}{key}"
            val = row.get(col)
            if pd.notna(val):
                lines.append(f"{name} {label}: {val}")
        linecards_col = f"{prefix}solution_linecards"
        linecards_val = row.get(linecards_col)
        if pd.notna(linecards_val):
            lines.append(f"{name} linecards: {_wrap_linecards(linecards_val)}")
        return lines

    speed_lines = []
    for col in speed_columns:
        val = row.get(col)
        if pd.notna(val) and val != 0:
            speed_lines.append(f"{col}: {int(val)}")

    if speed_lines:
        speed_block = "ports by speed: " + ", ".join(speed_lines)
    else:
        speed_block = "ports by speed: 0"

    return "<br>".join(
        _vendor_block(vendor1, "vendor1_cost", "v1_")
        + _vendor_block(vendor2, "vendor2_cost", "v2_")
        + [speed_block]
    )


def _render_vendor_comparison(vendor1, vendor2):
    df1 = _load_results(result_paths_by_vendor[vendor1])
    df2 = _load_results(result_paths_by_vendor[vendor2])
    speed_columns = _get_speed_columns(df1)
    compare_df = _build_compare_df(df1, df2, speed_columns)

    def _make_plot(**thresholds):
        filtered = compare_df
        for col, min_val in thresholds.items():
            filtered = filtered[filtered[col] >= min_val]

        green_x, green_y, red_x, red_y = [], [], [], []
        star_x, star_y, cross_x, cross_y = [], [], [], []
        green_text, red_text, star_text, cross_text = [], [], [], []

        for _, row in filtered.iterrows():
            bw = row["total_bandwidth"]
            cost1 = row["vendor1_cost"]
            cost2 = row["vendor2_cost"]
            hover_text = _make_hover_text(row, vendor1, vendor2, speed_columns)

            if pd.notna(cost1) and pd.notna(cost2):
                if cost1 <= cost2:
                    green_x.extend([bw, bw, None])
                    green_y.extend([cost1, cost2, None])
                    green_text.extend([hover_text, hover_text, None])
                else:
                    red_x.extend([bw, bw, None])
                    red_y.extend([cost1, cost2, None])
                    red_text.extend([hover_text, hover_text, None])
            elif pd.notna(cost1) and pd.isna(cost2):
                star_x.append(bw)
                star_y.append(cost1)
                star_text.append(hover_text)
            elif pd.isna(cost1) and pd.notna(cost2):
                cross_x.append(bw)
                cross_y.append(cost2)
                cross_text.append(hover_text)

        fig = go.Figure()
        if green_x:
            fig.add_trace(go.Scatter(
                x=green_x,
                y=green_y,
                mode="lines",
                line=dict(color="rgba(0, 128, 0, 0.4)", width=2),
                name=f"{vendor1} wins",
                hovertext=green_text,
                hoverinfo="text",
            ))
        if red_x:
            fig.add_trace(go.Scatter(
                x=red_x,
                y=red_y,
                mode="lines",
                line=dict(color="rgba(255, 0, 0, 0.4)", width=2),
                name=f"{vendor2} wins",
                hovertext=red_text,
                hoverinfo="text",
            ))
        if star_x:
            fig.add_trace(go.Scatter(
                x=star_x,
                y=star_y,
                mode="markers",
                marker=dict(color="green", symbol="star", size=10),
                name=f"{vendor1} only",
                hovertext=star_text,
                hoverinfo="text",
            ))
        if cross_x:
            fig.add_trace(go.Scatter(
                x=cross_x,
                y=cross_y,
                mode="markers",
                marker=dict(color="red", symbol="x", size=10),
                name=f"{vendor2} only",
                hovertext=cross_text,
                hoverinfo="text",
            ))

        fig.update_layout(
            title=f"{vendor1} vs {vendor2} (row-by-row costs)",
            hoverlabel=dict(font_size=10),
            height=500,
        )
        fig.update_xaxes(type="log", title="total_bandwidth")
        fig.update_yaxes(type="log", title="solution_cost")
        fig.write_html(f"{vendor1}_vs_{vendor2}_comparison.html")
        fig.show()

    sliders = {}
    for col in speed_columns:
        if col in compare_df.columns:
            max_val = int(compare_df[col].max()) if compare_df[col].notna().any() else 0
            sliders[col] = widgets.IntSlider(
                value=0,
                min=0,
                max=max_val,
                step=1,
                description=str(col),
                continuous_update=False,
                orientation="vertical",
                layout=widgets.Layout(width="50px", height="220px"),
                style={"description_width": "0px"},
            )

    ui = widgets.HBox(
        list(sliders.values()),
        layout=widgets.Layout(
            align_items="flex-end",
            flex_flow="row wrap",
        ),
    )
    out = widgets.interactive_output(_make_plot, sliders)

    container = widgets.VBox([
        ui,
        out,
    ])

    display(container)


_render_vendor_comparison(vendor1, vendor2)

